# TennisMyLife — Gallica ALTO + RapidOCR batch worker

Interactive Colab batch processor. It reads a manifest snapshot, processes each row independently, checkpoints locally, and creates a result bundle for import on the VPS.

Modes per row:
- `ALTO`: download and validate Gallica ALTO XML.
- `RAPID`: download `.highres`, run RapidOCR (`HQ` or `STANDARD`), write compatible JSON/TXT/hits.

This notebook is designed for manually started batch sessions, not as a permanent background worker.


In [ ]:
!pip -q install rapidocr==3.9.2 onnxruntime==1.29.0 opencv-python-headless==4.12.0.88 Pillow==11.3.0 paramiko==3.5.1


In [ ]:
import os, re, csv, json, time, tarfile, shutil, getpass, urllib.request, urllib.parse, urllib.error
from pathlib import Path
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor, as_completed
import xml.etree.ElementTree as ET

ROOT=Path('/content/tml_colab')
ROOT.mkdir(exist_ok=True)
MANIFEST=ROOT/'manifest.tsv'
OUT=ROOT/'out'
OUT.mkdir(exist_ok=True)
IMAGES=ROOT/'images'
IMAGES.mkdir(exist_ok=True)
print(ROOT)


In [ ]:
# Optional direct VPS snapshot download.
# Recommended: create Colab secrets VPS_HOST, VPS_USER and VPS_KEY (private key text).
# If unavailable, this cell falls back to manual manifest upload.
USE_SFTP=True
REMOTE_MANIFEST='/home/andre/GallicaJobs/_colab/outgoing/current/manifest.tsv'

def load_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if USE_SFTP:
    host=load_secret('VPS_HOST'); user=load_secret('VPS_USER'); keytxt=load_secret('VPS_KEY')
    if host and user and keytxt:
        import paramiko, io
        key=paramiko.RSAKey.from_private_key(io.StringIO(keytxt))
        tr=paramiko.Transport((host,22)); tr.connect(username=user,pkey=key)
        sftp=paramiko.SFTPClient.from_transport(tr)
        sftp.get(REMOTE_MANIFEST,str(MANIFEST))
        sftp.close(); tr.close()
        print('manifest downloaded from VPS')
    else:
        USE_SFTP=False

if not MANIFEST.exists():
    from google.colab import files
    up=files.upload()
    src=next(iter(up))
    shutil.copy2(src,MANIFEST)
    print('uploaded',src)


In [ ]:
def read_manifest():
    with MANIFEST.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f,delimiter='\t'))
    seen=set(); out=[]
    for r in rows:
        k=(r['ark'],str(int(float(r['page']))))
        if k in seen: continue
        seen.add(k); r['page']=k[1]; out.append(r)
    return out

rows=read_manifest()
from collections import Counter
print('rows',len(rows),Counter(r.get('mode','') for r in rows),Counter(r.get('profile','') for r in rows))


In [ ]:
UA='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/140 Safari/537.36'
KEYS=('tennis','lawntennis','championnat','championship','tournament','tournoi','raquette','singles','simple','messieurs','gentlemen','racingclub','puteaux','burdigala','primrose','dinard')

def stem(r): return f"{r['ark']}_f{int(r['page'])}"
def norm(s): return re.sub(r'[^a-z]','',s.lower())
def hit(s):
    n=norm(s)
    if any(k in n for k in KEYS): return True
    return max((SequenceMatcher(None,n,k).ratio() for k in KEYS),default=0)>=0.72

def alto_text(payload):
    root=ET.fromstring(payload); lines=[]
    for tl in root.iter():
        if tl.tag.split('}')[-1]!='TextLine': continue
        words=[]
        for x in tl.iter():
            if x.tag.split('}')[-1]=='String':
                t=(x.attrib.get('SUBS_CONTENT') or x.attrib.get('CONTENT') or '').strip()
                if t: words.append(t)
        if words: lines.append(' '.join(words))
    return '\n'.join(lines).strip()

def fetch(url,accept='*/*',attempts=5):
    last=None
    for a in range(attempts):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':UA,'Accept':accept,'Referer':'https://gallica.bnf.fr/'})
            with urllib.request.urlopen(req,timeout=60) as resp:
                return resp.status,resp.read()
        except urllib.error.HTTPError as e:
            last=e
            if e.code in (404,410): return e.code,b''
            time.sleep(min(15*(a+1),60))
        except Exception as e:
            last=e; time.sleep(min(15*(a+1),60))
    raise last


In [ ]:
def process_alto(r):
    base=OUT/stem(r)
    if base.with_suffix('.alto.xml').exists() and base.with_suffix('.txt').exists():
        return 'cached',stem(r)
    q=urllib.parse.urlencode({'O':r['ark'],'E':'ALTO','Deb':r['page']})
    code,payload=fetch('https://gallica.bnf.fr/RequestDigitalElement?'+q,'application/xml,text/xml;q=0.9,*/*;q=0.8')
    if code!=200 or not payload: return f'http_{code}',stem(r)
    text=alto_text(payload)
    if not text: return 'empty',stem(r)
    base.with_suffix('.alto.xml').write_bytes(payload)
    base.with_suffix('.txt').write_text(text+'\n',encoding='utf-8')
    meta={'ark':r['ark'],'page':r['page'],'mode':'ALTO','profile':'ALTO_NATIVE','status':'ok'}
    base.with_suffix('.meta.json').write_text(json.dumps(meta),encoding='utf-8')
    return 'ok',stem(r)

for i,r in enumerate([x for x in rows if x.get('mode')=='ALTO'],1):
    try: st,s=process_alto(r)
    except Exception as e: st,s=f'err:{type(e).__name__}',stem(r)
    if i<=10 or i%10==0 or st not in ('ok','cached'): print('ALTO',i,st,s,flush=True)
    time.sleep(1.0)


In [ ]:
# RapidOCR worker. CPU only.
os.environ.update({'OMP_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
_ENGINE={}
def get_engine(profile):
    if profile not in _ENGINE:
        from rapidocr import RapidOCR, LangRec, ModelType, OCRVersion
        p={
          'Global.log_level':'error',
          'EngineConfig.onnxruntime.intra_op_num_threads':1,
          'EngineConfig.onnxruntime.inter_op_num_threads':1,
          'Rec.lang_type':LangRec.LATIN,'Rec.model_type':ModelType.MOBILE,'Rec.ocr_version':OCRVersion.PPOCRV5,
          'Det.model_type':ModelType.SMALL,'Det.ocr_version':OCRVersion.PPOCRV6,
        }
        if profile=='HQ': p.update({'Det.limit_side_len':2048,'Det.limit_type':'min','Det.box_thresh':0.30,'Det.max_candidates':4000})
        else: p.update({'Det.limit_side_len':1536,'Det.limit_type':'min','Det.box_thresh':0.35,'Det.max_candidates':3000})
        _ENGINE[profile]=RapidOCR(params=p)
    return _ENGINE[profile]

def rapid_one(r):
    base=OUT/stem(r)
    if base.with_suffix('.json').exists() and base.with_suffix('.txt').exists():
        return 'cached',stem(r),0
    url=f"https://gallica.bnf.fr/ark:/12148/{r['ark']}/f{int(r['page'])}.highres"
    code,data=fetch(url,'image/*,*/*;q=0.8')
    if code!=200 or len(data)<10000 or data[:2]!=b'\xff\xd8': return f'img_{code}',stem(r),0
    ip=IMAGES/f"{stem(r)}.jpg"; ip.write_bytes(data)
    profile=(r.get('profile') or 'HQ').upper()
    t0=time.time(); res=get_engine(profile)(str(ip)); rows2=[]
    boxes=res.boxes if res.boxes is not None else []; txts=res.txts if res.txts is not None else []; scores=res.scores if res.scores is not None else []
    for b,t,s in zip(boxes,txts,scores):
        xs=[float(q[0]) for q in b]; ys=[float(q[1]) for q in b]
        rows2.append({'text':str(t),'score':float(s),'x1':min(xs),'x2':max(xs),'y1':min(ys),'y2':max(ys),'cx':sum(xs)/4,'cy':sum(ys)/4})
    rows2.sort(key=lambda z:(z['y1'],z['x1'])); texts=[z['text'] for z in rows2]; hits=[i for i,t in enumerate(texts) if hit(t)]
    base.with_suffix('.json').write_text(json.dumps({'source':url,'ocr_profile':profile,'ocr_generation':'colab_batch','rows':rows2,'hit_indices':hits},ensure_ascii=False),encoding='utf-8')
    base.with_suffix('.txt').write_text('\n'.join(texts)+'\n',encoding='utf-8')
    ctx=[]
    for i in hits:
        a=max(0,i-5);b=min(len(texts),i+13);ctx.append(f'--- {a+1}-{b} ---');ctx.extend(texts[a:b])
    base.with_suffix('.hits.txt').write_text('\n'.join(ctx)+'\n' if ctx else '',encoding='utf-8')
    ip.unlink(missing_ok=True)
    return 'ok',stem(r),time.time()-t0


In [ ]:
# Conservative default: 2 CPU OCR workers. Increase only if the assigned Colab runtime has spare CPUs/RAM.
RAPID_WORKERS=2
rapid=[r for r in rows if r.get('mode')=='RAPID']
with ProcessPoolExecutor(max_workers=RAPID_WORKERS) as ex:
    fut={ex.submit(rapid_one,r):r for r in rapid}
    done=0
    for f in as_completed(fut):
        done+=1
        try: st,s,sec=f.result()
        except Exception as e: st,s,sec=f'err:{type(e).__name__}',stem(fut[f]),0
        if done<=10 or done%10==0 or st not in ('ok','cached'): print('RAPID',done,'/',len(rapid),st,s,f'{sec:.1f}s',flush=True)


In [ ]:
# Build resumable result bundle.
stamp=time.strftime('%Y%m%d_%H%M%S')
bundle=ROOT/f'colab_result_{stamp}.tar.gz'
with tarfile.open(bundle,'w:gz') as tf:
    tf.add(OUT,arcname='out')
    tf.add(MANIFEST,arcname='manifest.tsv')
print(bundle, bundle.stat().st_size)


In [ ]:
# Optional direct upload of the finished bundle to VPS.
REMOTE_INCOMING='/home/andre/GallicaJobs/_colab/incoming'
if USE_SFTP:
    host=load_secret('VPS_HOST'); user=load_secret('VPS_USER'); keytxt=load_secret('VPS_KEY')
    import paramiko, io
    key=paramiko.RSAKey.from_private_key(io.StringIO(keytxt))
    tr=paramiko.Transport((host,22)); tr.connect(username=user,pkey=key)
    sftp=paramiko.SFTPClient.from_transport(tr)
    remote=f"{REMOTE_INCOMING}/{bundle.name}"
    sftp.put(str(bundle),remote)
    sftp.close(); tr.close()
    print('uploaded',remote)
else:
    from google.colab import files
    files.download(str(bundle))
